# SIM Expiry Automation - Initial Join EDA

**Purpose:** Explore the 3 source CSV files and understand how they join together.

**Join Logic:**
1. Base table: `conversations` (filtered by agent_id)
2. LEFT JOIN `kpi_results` on `conversation_id = voiceConversationId`
3. LEFT JOIN `twilio_webhook_events` on `conversation_id`

---

In [1]:
import pandas as pd
import json
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Load Raw CSVs

In [2]:
# File paths
DATA_DIR = Path('../data')

conversations = pd.read_csv(DATA_DIR / 'conversations.csv')
kpi_results = pd.read_csv(DATA_DIR / 'kpi_results.csv')
twilio_events = pd.read_csv(DATA_DIR / 'twilio_webhook_events.csv')

print(f"Conversations shape: {conversations.shape}")
print(f"KPI Results shape: {kpi_results.shape}")
print(f"Twilio Events shape: {twilio_events.shape}")

Conversations shape: (1834, 14)
KPI Results shape: (915, 18)
Twilio Events shape: (128, 5)


## 2. Explore Table Structures

In [3]:
print("\n=== CONVERSATIONS Columns ===")
print(conversations.columns.tolist())
print(f"\nData types:")
conversations.info()


=== CONVERSATIONS Columns ===
['conversation_id', 'start_timestamp', 'end_timestamp', 'call_config', 'call_logs', 'contact_number', 'status', 'failure_reason', 'results', 'telephony_id', 'agent_id', 'agent_version_id', 'agent_schedule_metadata_id', 'tag']

Data types:
<class 'pandas.DataFrame'>
RangeIndex: 1834 entries, 0 to 1833
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   conversation_id             1834 non-null   str    
 1   start_timestamp             1834 non-null   int64  
 2   end_timestamp               1441 non-null   float64
 3   call_config                 1834 non-null   str    
 4   call_logs                   1440 non-null   str    
 5   contact_number              1834 non-null   str    
 6   status                      1834 non-null   str    
 7   failure_reason              0 non-null      float64
 8   results                     0 non-null      float64
 9   te

In [4]:
print("\n=== KPI_RESULTS Columns ===")
print(kpi_results.columns.tolist())
print(f"\nData types:")
kpi_results.info()


=== KPI_RESULTS Columns ===
['id', 'workspaceId', 'outputJson', 'createdAt', 'agentType', 'chatbotAgentId', 'voiceAgentId', 'agenticFlowId', 'customAgentId', 'chatbotAgentVersionId', 'voiceAgentVersionId', 'agenticFlowVersionId', 'customAgentVersionId', 'chatbotConversationId', 'voiceConversationId', 'agenticFlowConversationId', 'customAgentConversationId', 'aiCoworkerRoleId']

Data types:
<class 'pandas.DataFrame'>
RangeIndex: 915 entries, 0 to 914
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         915 non-null    int64  
 1   workspaceId                915 non-null    str    
 2   outputJson                 915 non-null    str    
 3   createdAt                  915 non-null    str    
 4   agentType                  915 non-null    str    
 5   chatbotAgentId             89 non-null     str    
 6   voiceAgentId               826 non-null    float64
 7   age

In [5]:
print("\n=== TWILIO_WEBHOOK_EVENTS Columns ===")
print(twilio_events.columns.tolist())
print(f"\nData types:")
twilio_events.info()


=== TWILIO_WEBHOOK_EVENTS Columns ===
['id', 'call_sid', 'sip_code', 'event', 'conversation_id']

Data types:
<class 'pandas.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               128 non-null    int64
 1   call_sid         128 non-null    str  
 2   sip_code         128 non-null    int64
 3   event            128 non-null    str  
 4   conversation_id  128 non-null    str  
dtypes: int64(2), str(3)
memory usage: 5.1 KB


In [6]:
# Preview first few rows
print("\n=== CONVERSATIONS Sample ===")
conversations.head(3)


=== CONVERSATIONS Sample ===


,conversation_id,start_timestamp,end_timestamp,call_config,call_logs,contact_number,status,failure_reason,results,telephony_id,agent_id,agent_version_id,agent_schedule_metadata_id,tag
0,1d6dce32bcd94121bbcb9ff8752690e8,1768367605229,NaN,"""{\""customer_phone\"": \""+639605268729\""}""",NaN,+639605268729,in_progress,NaN,NaN,CAa308c19dbdd14e2acbefc295b3aaf57d,1,NaN,NaN,user
1,2e71e337cb0e46f4b522a260d6f0c671,1768367670664,NaN,"""{\""customer_phone\"": \""+639605268729\""}""",NaN,+639605268729,in_progress,NaN,NaN,CA07b3aa122d0055f71fcf4ff370d1dab6,1,NaN,NaN,user
2,47f6efccb9c74c3087443ad11a02e67c,1768367697549,NaN,"""{\""customer_phone\"": \""+639060269554\""}""",NaN,+639060269554,in_progress,NaN,NaN,CAfd1caf2c1c27126214aa0bad411fd8ec,1,NaN,NaN,user


In [7]:
print("\n=== KPI_RESULTS Sample ===")
kpi_results.head(3)


=== KPI_RESULTS Sample ===


,id,workspaceId,outputJson,createdAt,agentType,chatbotAgentId,voiceAgentId,agenticFlowId,customAgentId,chatbotAgentVersionId,voiceAgentVersionId,agenticFlowVersionId,customAgentVersionId,chatbotConversationId,voiceConversationId,agenticFlowConversationId,customAgentConversationId,aiCoworkerRoleId
0,1,cmkap74u500022flp24p2sge9,"{""user_sentiment"": 3, ""is_appointment_booked"": false}",2026-01-14 05:25:03.07,voicebot,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,c0ba55f5ed5f40c29dd41e18d8560f21,NaN,NaN,NaN
1,34,cmosf1u2w000l3b5k3xfcgc38,"{""user_sentiment"": 2, ""is_appointment_booked"": false}",2026-05-05 09:26:44.877,voicebot,NaN,798.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cb3f96ee5ea44f26ba5c4ed114728557,NaN,NaN,NaN
2,35,cmkap74u500022flp24p2sge9,"{""call_completed"": false, ""question_topics"": [], ""customer_disposition"": ""neutral"", ""non_retenti...",2026-05-05 09:34:20.564,voicebot,NaN,793.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72e55c66392e44a7ae7ce0b98b030476,NaN,NaN,NaN


In [8]:
print("\n=== TWILIO_EVENTS Sample ===")
twilio_events.head(3)


=== TWILIO_EVENTS Sample ===


,id,call_sid,sip_code,event,conversation_id
0,1,CAa308c19dbdd14e2acbefc295b3aaf57d,480,"{""failed"": {""To"": ""+639605268729"", ""From"": ""+639272797345"", ""ToZip"": """", ""Called"": ""+63960526872...",1d6dce32bcd94121bbcb9ff8752690e8
1,2,CA07b3aa122d0055f71fcf4ff370d1dab6,480,"{""failed"": {""To"": ""+639605268729"", ""From"": ""+639272797345"", ""ToZip"": """", ""Called"": ""+63960526872...",2e71e337cb0e46f4b522a260d6f0c671
2,3,CAfd1caf2c1c27126214aa0bad411fd8ec,480,"{""failed"": {""To"": ""+639060269554"", ""From"": ""+639272797345"", ""ToZip"": """", ""Called"": ""+63906026955...",47f6efccb9c74c3087443ad11a02e67c


## 3. Check Join Keys

In [9]:
# Check conversation_id uniqueness in conversations table
print("=== CONVERSATIONS.conversation_id ===")
print(f"Total rows: {len(conversations)}")
print(f"Unique conversation_ids: {conversations['conversation_id'].nunique()}")
print(f"Duplicates: {len(conversations) - conversations['conversation_id'].nunique()}")
print(f"Nulls: {conversations['conversation_id'].isna().sum()}")

=== CONVERSATIONS.conversation_id ===
Total rows: 1834
Unique conversation_ids: 1834
Duplicates: 0
Nulls: 0


In [10]:
# Check kpi_results join key
print("\n=== KPI_RESULTS.voiceConversationId ===")
print(f"Total rows: {len(kpi_results)}")
print(f"Unique voiceConversationIds: {kpi_results['voiceConversationId'].nunique()}")
print(f"Duplicates: {len(kpi_results) - kpi_results['voiceConversationId'].nunique()}")
print(f"Nulls: {kpi_results['voiceConversationId'].isna().sum()}")


=== KPI_RESULTS.voiceConversationId ===
Total rows: 915
Unique voiceConversationIds: 821
Duplicates: 94
Nulls: 89


In [37]:
kpi_results[
    kpi_results.duplicated(subset=['voiceConversationId'], keep=False) &
    (kpi_results['voiceAgentId'] == 1060.0)
].sort_values(by='voiceConversationId')

,id,workspaceId,outputJson,createdAt,agentType,chatbotAgentId,voiceAgentId,agenticFlowId,customAgentId,chatbotAgentVersionId,voiceAgentVersionId,agenticFlowVersionId,customAgentVersionId,chatbotConversationId,voiceConversationId,agenticFlowConversationId,customAgentConversationId,aiCoworkerRoleId


In [11]:
# Check twilio_events join key
print("\n=== TWILIO_EVENTS.conversation_id ===")
print(f"Total rows: {len(twilio_events)}")
print(f"Unique conversation_ids: {twilio_events['conversation_id'].nunique()}")
print(f"Duplicates: {len(twilio_events) - twilio_events['conversation_id'].nunique()}")
print(f"Nulls: {twilio_events['conversation_id'].isna().sum()}")


=== TWILIO_EVENTS.conversation_id ===
Total rows: 128
Unique conversation_ids: 128
Duplicates: 0
Nulls: 0


In [12]:
# Check join key overlap
conv_ids = set(conversations['conversation_id'].dropna())
kpi_ids = set(kpi_results['voiceConversationId'].dropna())
twilio_ids = set(twilio_events['conversation_id'].dropna())

print("\n=== Join Key Overlap ===")
print(f"Conversations IDs: {len(conv_ids):,}")
print(f"KPI Results IDs: {len(kpi_ids):,}")
print(f"Twilio Events IDs: {len(twilio_ids):,}")
print(f"\nOverlap:")
print(f"  Conversations ∩ KPI: {len(conv_ids & kpi_ids):,} ({len(conv_ids & kpi_ids)/len(conv_ids)*100:.1f}%)")
print(f"  Conversations ∩ Twilio: {len(conv_ids & twilio_ids):,} ({len(conv_ids & twilio_ids)/len(conv_ids)*100:.1f}%)")
print(f"  All 3 tables: {len(conv_ids & kpi_ids & twilio_ids):,}")


=== Join Key Overlap ===
Conversations IDs: 1,834
KPI Results IDs: 821
Twilio Events IDs: 128

Overlap:
  Conversations ∩ KPI: 821 (44.8%)
  Conversations ∩ Twilio: 128 (7.0%)
  All 3 tables: 2


## 4. Explore Agent Distribution

In [39]:
print("=== Agent Distribution ===")
print("\nConversations by agent_id:")
print(conversations['agent_id'].value_counts().sort_values(ascending=False))

=== Agent Distribution ===

Conversations by agent_id:
agent_id
826     478
1060    388
37      131
795     100
628      97
563      75
794      51
925      42
796      40
595      39
793      35
1189     29
34       28
727      27
167      24
1058     24
2        22
331      21
829      17
694      12
959      11
1123     11
1061     10
68        9
1255      9
35        9
1156      8
100       7
760       7
859       7
1057      7
36        5
298       5
397       5
892       5
828       4
1090      4
1043      4
463       4
1         4
431       3
496       3
562       3
38        2
411       2
364       1
406       1
529       1
798       1
958       1
1222      1
Name: count, dtype: int64


In [40]:
print("\nKPI Results by voiceAgentId:")
print(kpi_results['voiceAgentId'].value_counts().sort_values(ascending=False))


KPI Results by voiceAgentId:
voiceAgentId
1060.0    279
826.0     244
795.0      54
628.0      37
794.0      29
563.0      25
1189.0     25
925.0      21
1058.0     16
796.0      11
959.0       9
1123.0      9
1156.0      9
1255.0      9
829.0       8
859.0       7
892.0       5
1061.0      5
828.0       4
1043.0      4
1090.0      4
1057.0      3
793.0       2
1222.0      2
1.0         1
798.0       1
760.0       1
34.0        1
958.0       1
Name: count, dtype: int64


In [15]:
# Check if twilio_events has agent info
print("\nTwilio Events columns (checking for agent info):")
print([col for col in twilio_events.columns if 'agent' in col.lower()])


Twilio Events columns (checking for agent info):
[]


## 5. Preview JSON Columns

Several columns contain JSON data that needs parsing.

In [16]:
# Sample call_logs JSON from conversations
print("=== CONVERSATIONS.call_logs (sample) ===")
sample_call_logs = conversations[conversations['call_logs'].notna()]['call_logs'].iloc[0]
print(json.dumps(json.loads(sample_call_logs), indent=2)[:1000])  # First 1000 chars

=== CONVERSATIONS.call_logs (sample) ===
[
  {
    "sender": "bot",
    "timestamp": 1768368275.515775,
    "stage_details": {
      "stage_name": "initial response handler",
      "stage_id": "f87a3d0f-0c66-4c1b-8f30-e082297be365",
      "stage_type": "start"
    },
    "text": "BOT: Hi!  BrightSmile Dent al Care.  I am  Claire -  How can I help you today?  ",
    "is_final": true,
    "is_backchannel": false,
    "is_end_of_turn": true
  },
  {
    "sender": "human",
    "timestamp": 1768368287.746283,
    "stage_details": {
      "stage_name": "initial response handler",
      "stage_id": "f87a3d0f-0c66-4c1b-8f30-e082297be365",
      "stage_type": "start"
    },
    "text": "HUMAN: Hi, can I set an appointment? Masakit ng ipin ko.",
    "is_final": false,
    "is_backchannel": false,
    "is_end_of_turn": false
  },
  {
    "sender": "human",
    "timestamp": 1768368297.2894504,
    "stage_details": {
      "stage_name": "initial response handler",
      "stage_id": "f87a3d0f-0c66-4

In [17]:
# Sample outputJson from kpi_results
print("\n=== KPI_RESULTS.outputJson (sample) ===")
sample_output = kpi_results[kpi_results['outputJson'].notna()]['outputJson'].iloc[0]
print(json.dumps(json.loads(sample_output), indent=2))


=== KPI_RESULTS.outputJson (sample) ===
{
  "user_sentiment": 3,
  "is_appointment_booked": false
}


In [18]:
# Sample event JSON from twilio_events
print("\n=== TWILIO_EVENTS.event (sample) ===")
sample_event = twilio_events[twilio_events['event'].notna()]['event'].iloc[0]
print(json.dumps(json.loads(sample_event), indent=2)[:1500])  # First 1500 chars


=== TWILIO_EVENTS.event (sample) ===
{
  "failed": {
    "To": "+639605268729",
    "From": "+639272797345",
    "ToZip": "",
    "Called": "+639605268729",
    "Caller": "+639272797345",
    "ToCity": "",
    "CallSid": "CAa308c19dbdd14e2acbefc295b3aaf57d",
    "FromZip": "",
    "ToState": "",
    "Duration": "0",
    "FromCity": "",
    "CalledZip": "",
    "CallerZip": "",
    "Direction": "outbound-api",
    "FromState": "",
    "Timestamp": "Wed, 14 Jan 2026 05:13:25 +0000",
    "ToCountry": "PH",
    "AccountSid": "AC303f8edc4d3a765f5efb64db7c7afb8d",
    "ApiVersion": "2010-04-01",
    "CallStatus": "failed",
    "CalledCity": "",
    "CallerCity": "",
    "CalledState": "",
    "CallerState": "",
    "FromCountry": "PH",
    "CallDuration": "0",
    "CalledCountry": "PH",
    "CallerCountry": "PH",
    "CallbackSource": "call-progress-events",
    "SequenceNumber": "1",
    "SipResponseCode": "480"
  },
  "initiated": {
    "To": "+639605268729",
    "From": "+639272797345",


## 6. Filter to Target Agent (1060)

Following the repository logic, filter to agent_id = 1060 first.

In [19]:
AGENT_ID = 1060

# Filter conversations
conv_filtered = conversations[conversations['agent_id'] == AGENT_ID].copy()
print(f"Conversations for agent {AGENT_ID}: {len(conv_filtered):,} rows")

# Filter kpi_results
kpi_filtered = kpi_results[kpi_results['voiceAgentId'] == AGENT_ID].copy()
print(f"KPI Results for agent {AGENT_ID}: {len(kpi_filtered):,} rows")

Conversations for agent 1060: 388 rows
KPI Results for agent 1060: 279 rows


## 7. Perform Joins (Step-by-Step)

In [20]:
# Step 1: LEFT JOIN conversations with kpi_results
print("=== Step 1: Join Conversations + KPI Results ===")

# Rename join key in kpi_results for clarity
kpi_for_join = kpi_filtered.rename(columns={'voiceConversationId': 'conversation_id'})

merged_step1 = conv_filtered.merge(
    kpi_for_join,
    on='conversation_id',
    how='left',
    suffixes=('', '_kpi')
)

print(f"Conversations rows: {len(conv_filtered):,}")
print(f"After joining KPI: {len(merged_step1):,}")
print(f"KPI match rate: {merged_step1['voiceAgentId'].notna().sum() / len(merged_step1) * 100:.1f}%")

=== Step 1: Join Conversations + KPI Results ===
Conversations rows: 388
After joining KPI: 388
KPI match rate: 71.9%


In [21]:
# Check for duplicates introduced by join
if len(merged_step1) != len(conv_filtered):
    print("\n⚠️ WARNING: Join introduced duplicates!")
    dup_ids = merged_step1[merged_step1.duplicated(subset='conversation_id', keep=False)]['conversation_id']
    print(f"Duplicate conversation_ids: {dup_ids.nunique()}")
else:
    print("\n✓ No duplicates introduced in Step 1")


✓ No duplicates introduced in Step 1


In [22]:
# Step 2: LEFT JOIN with twilio_events
print("\n=== Step 2: Join + Twilio Events ===")

merged_final = merged_step1.merge(
    twilio_events,
    on='conversation_id',
    how='left',
    suffixes=('', '_twilio')
)

print(f"After Step 1: {len(merged_step1):,} rows")
print(f"After joining Twilio: {len(merged_final):,} rows")
print(f"Twilio match rate: {merged_final['event'].notna().sum() / len(merged_final) * 100:.1f}%")


=== Step 2: Join + Twilio Events ===
After Step 1: 388 rows
After joining Twilio: 388 rows
Twilio match rate: 0.0%


In [23]:
# Check for duplicates introduced by join
if len(merged_final) != len(merged_step1):
    print("\n⚠️ WARNING: Join introduced duplicates!")
    dup_ids = merged_final[merged_final.duplicated(subset='conversation_id', keep=False)]['conversation_id']
    print(f"Duplicate conversation_ids: {dup_ids.nunique()}")
else:
    print("\n✓ No duplicates introduced in Step 2")


✓ No duplicates introduced in Step 2


## 8. Examine Joined Table

In [24]:
print("=== Final Joined Table ===")
print(f"Shape: {merged_final.shape}")
print(f"\nColumns ({len(merged_final.columns)}):")
print(merged_final.columns.tolist())

=== Final Joined Table ===
Shape: (388, 35)

Columns (35):
['conversation_id', 'start_timestamp', 'end_timestamp', 'call_config', 'call_logs', 'contact_number', 'status', 'failure_reason', 'results', 'telephony_id', 'agent_id', 'agent_version_id', 'agent_schedule_metadata_id', 'tag', 'id', 'workspaceId', 'outputJson', 'createdAt', 'agentType', 'chatbotAgentId', 'voiceAgentId', 'agenticFlowId', 'customAgentId', 'chatbotAgentVersionId', 'voiceAgentVersionId', 'agenticFlowVersionId', 'customAgentVersionId', 'chatbotConversationId', 'agenticFlowConversationId', 'customAgentConversationId', 'aiCoworkerRoleId', 'id_twilio', 'call_sid', 'sip_code', 'event']


In [25]:
# Check null rates for key joined columns
print("\n=== Null Rates in Joined Columns ===")
joined_cols = ['outputJson', 'voiceAgentId', 'event', 'created_at']
available_cols = [col for col in joined_cols if col in merged_final.columns]

null_rates = merged_final[available_cols].isna().sum() / len(merged_final) * 100
null_rates = null_rates.sort_values(ascending=False)
print(null_rates.to_string())


=== Null Rates in Joined Columns ===
event           100.000000
outputJson       28.092784
voiceAgentId     28.092784


In [26]:
# Sample rows from joined table
print("\n=== Sample Joined Rows ===")
merged_final.head(3)


=== Sample Joined Rows ===


,conversation_id,start_timestamp,end_timestamp,call_config,call_logs,contact_number,status,failure_reason,results,telephony_id,agent_id,agent_version_id,agent_schedule_metadata_id,tag,id,workspaceId,outputJson,createdAt,agentType,chatbotAgentId,voiceAgentId,agenticFlowId,customAgentId,chatbotAgentVersionId,voiceAgentVersionId,agenticFlowVersionId,customAgentVersionId,chatbotConversationId,agenticFlowConversationId,customAgentConversationId,aiCoworkerRoleId,id_twilio,call_sid,sip_code,event
0,f27af08681924831ba534067494d4a5f,1781057506586,1.781029e+12,{},"{""transcript"": [{""turn_id"": ""b97c881a-cc54-4eb8-b067-221451c496d6"", ""role"": ""user"", ""content"": ""...",+639177916020,completed,NaN,NaN,058021ff-1c3e-4168-9c11-84bf8d33e75c,1060,NaN,NaN,NaN,656.0,cmnfqhmzn0050t5yfwd3eb4j4,"{""call_completed"": true, ""question_topics"": [""expiry_info""], ""customer_disposition"": ""skeptic"", ...",2026-06-10 10:12:46.074,voicebot,NaN,1060.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6a2e99b3a01d4481987f8820be831401,1781057608623,NaN,{},NaN,+639760380854,in_progress,NaN,NaN,cb67781e-7222-4b06-80a3-81c6da1baa94,1060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0fa01a3252f34e08a47555b4fc18b246,1781057695808,1.781029e+12,{},"{""transcript"": [{""turn_id"": ""83510c5b-ea8b-488e-ab69-de0351d3b491"", ""role"": ""user"", ""content"": ""...",+639760380854,completed,NaN,NaN,bff23f24-ab82-4fc6-bef5-c17bb5c64e4d,1060,NaN,NaN,NaN,657.0,cmnfqhmzn0050t5yfwd3eb4j4,"{""call_completed"": true, ""question_topics"": [], ""customer_disposition"": ""disengaged"", ""non_reten...",2026-06-10 10:16:02.259,voicebot,NaN,1060.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# Show a row that has all 3 sources (if any exist)
print("\n=== Sample Row with All 3 Sources ===")
has_all = merged_final[
    (merged_final['voiceAgentId'].notna()) & 
    (merged_final['event'].notna())
]

if len(has_all) > 0:
    print(f"Found {len(has_all)} rows with data from all 3 sources")
    has_all.head(1).T
else:
    print("No rows have data from all 3 sources")


=== Sample Row with All 3 Sources ===
No rows have data from all 3 sources


## 9. Summary Statistics

In [28]:
print("=== Join Summary for Agent 1060 ===")
print(f"\nBase (Conversations): {len(conv_filtered):,} rows")
print(f"\nMatches:")
print(f"  - KPI Results: {merged_final['voiceAgentId'].notna().sum():,} ({merged_final['voiceAgentId'].notna().sum() / len(merged_final) * 100:.1f}%)")
print(f"  - Twilio Events: {merged_final['event'].notna().sum():,} ({merged_final['event'].notna().sum() / len(merged_final) * 100:.1f}%)")
print(f"  - Both KPI + Twilio: {((merged_final['voiceAgentId'].notna()) & (merged_final['event'].notna())).sum():,}")
print(f"  - Neither: {((merged_final['voiceAgentId'].isna()) & (merged_final['event'].isna())).sum():,}")

=== Join Summary for Agent 1060 ===

Base (Conversations): 388 rows

Matches:
  - KPI Results: 279 (71.9%)
  - Twilio Events: 0 (0.0%)
  - Both KPI + Twilio: 0
  - Neither: 109


In [29]:
# Date range in conversations
if 'start_timestamp' in conv_filtered.columns:
    conv_filtered['start_dt'] = pd.to_datetime(conv_filtered['start_timestamp'], unit='ms', utc=True)
    print("\n=== Date Range in Conversations ===")
    print(f"Earliest: {conv_filtered['start_dt'].min()}")
    print(f"Latest: {conv_filtered['start_dt'].max()}")
    print(f"Days covered: {(conv_filtered['start_dt'].max() - conv_filtered['start_dt'].min()).days}")


=== Date Range in Conversations ===
Earliest: 2026-06-10 02:11:46.586000+00:00
Latest: 2026-07-03 03:50:05.406000+00:00
Days covered: 23


## 10. Save Merged Table (Optional)

Uncomment to save the joined table for further analysis.

In [ ]:
# merged_final.to_csv('../eda/merged_agent_1060.csv', index=False)
# print("Saved merged table to: ../eda/merged_agent_1060.csv")

In [63]:
# conversations.query("agent_id == 1060")

joined = pd.merge(
    conversations,
    twilio_events,
    left_on='telephony_id',
    right_on='call_sid',
    how='inner',
)

In [68]:
joined[['conversation_id_x', 'conversation_id_y', 'agent_id', 'call_sid', 'telephony_id', 'event']].head(5)

,conversation_id_x,conversation_id_y,agent_id,call_sid,telephony_id,event
0,1d6dce32bcd94121bbcb9ff8752690e8,1d6dce32bcd94121bbcb9ff8752690e8,1,CAa308c19dbdd14e2acbefc295b3aaf57d,CAa308c19dbdd14e2acbefc295b3aaf57d,"{""failed"": {""To"": ""+639605268729"", ""From"": ""+639272797345"", ""ToZip"": """", ""Called"": ""+63960526872..."
1,2e71e337cb0e46f4b522a260d6f0c671,2e71e337cb0e46f4b522a260d6f0c671,1,CA07b3aa122d0055f71fcf4ff370d1dab6,CA07b3aa122d0055f71fcf4ff370d1dab6,"{""failed"": {""To"": ""+639605268729"", ""From"": ""+639272797345"", ""ToZip"": """", ""Called"": ""+63960526872..."
2,47f6efccb9c74c3087443ad11a02e67c,47f6efccb9c74c3087443ad11a02e67c,1,CAfd1caf2c1c27126214aa0bad411fd8ec,CAfd1caf2c1c27126214aa0bad411fd8ec,"{""failed"": {""To"": ""+639060269554"", ""From"": ""+639272797345"", ""ToZip"": """", ""Called"": ""+63906026955..."
3,b26df5c454994ae0b81013115a561678,b26df5c454994ae0b81013115a561678,2,CA89e270a3b700b109087621db68360cf2,CA89e270a3b700b109087621db68360cf2,"{""failed"": {""To"": ""+639605268729"", ""From"": ""+639272797345"", ""ToZip"": """", ""Called"": ""+63960526872..."
4,e6fd6186ff5d40eda5ee476231d1a9b5,e6fd6186ff5d40eda5ee476231d1a9b5,2,CAfa6967f28f250ed8a6dbb500d675cb95,CAfa6967f28f250ed8a6dbb500d675cb95,"{""failed"": {""To"": ""+639605268729"", ""From"": ""+639272797345"", ""ToZip"": """", ""Called"": ""+63960526872..."


In [55]:
joined['agent_id'].value_counts().sort_values(ascending=False)

agent_id
37     32
2      22
331    21
34     18
298     5
397     5
1       4
463     4
496     3
431     3
411     2
595     2
628     2
35      1
364     1
406     1
529     1
563     1
Name: count, dtype: int64

In [71]:
conversations.query("agent_id == 1060")['status'].value_counts()

status
completed      280
in_progress    108
Name: count, dtype: int64

In [80]:
merged_step1['outputJson'].isna().sum()

np.int64(109)

In [82]:
merged_step1.query("outputJson.isna() & status != 'in_progress'")

,conversation_id,start_timestamp,end_timestamp,call_config,call_logs,contact_number,status,failure_reason,results,telephony_id,agent_id,agent_version_id,agent_schedule_metadata_id,tag,id,workspaceId,outputJson,createdAt,agentType,chatbotAgentId,voiceAgentId,agenticFlowId,customAgentId,chatbotAgentVersionId,voiceAgentVersionId,agenticFlowVersionId,customAgentVersionId,chatbotConversationId,agenticFlowConversationId,customAgentConversationId,aiCoworkerRoleId
342,ca5bc77e1d604a3ba8a6fa4ee41d1113,1782133590063,1.782134e+12,{},"{""transcript"": [{""turn_id"": ""3c6dba1c-57dd-4f95-8075-80e25c29e3ec"", ""role"": ""assistant"", ""conten...",+639524551037,completed,NaN,NaN,d43c88cb-14fe-4df2-be67-40ce3d003bb5,1060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [93]:
import ast
import pandas as pd

# Parse the string into an actual dict
parsed = twilio_events['event'].apply(ast.literal_eval)

In [94]:
unique_keys = parsed.apply(lambda d: list(d.keys())).explode().unique()

In [95]:
unique_keys

<StringArray>
[     'failed',   'initiated',     'ringing',   'completed', 'in-progress',
   'no-answer',        'busy']
Length: 7, dtype: str